In [ ]:
%load_ext autoreload
%autoreload 3 --print --log

# 本地 HM3D：数据集读取与语义融合

通过 `dataset_wrapper.datasets.hm3d.HM3D` 读取 `/R/Datasets/3DGS/hm3dv2` 的三个真实帧。
使用 `GRAPHMAP_HM3D_ROOT` / `GRAPHMAP_HM3D_SCENE` 修改路径；数据集负责索引、RGB/depth/semantic 读取和回放元数据，graphmap 负责坐标、投影与融合。

`load_replay=True` 启用 FNAV 导出的可信 `runtime_log.pkl` / `navigation_replay.json` 读取；位姿通过 `annotations["pose"]` 返回，元数据通过 `metadata` 返回。graphmap 示例不再解析这些文件。
已验证本机配置为 640×640、HFOV 64°、深度 uint16 毫米。导出位姿是 **floor-relative graphmap 相机位姿**，不再次叠加相机外参、减楼层高度或换 Habitat 轴。语义为实例 ID，缺失时为 None。

AIO 已配置本地 dataset_wrapper；其他环境先安装更新后的 `dataset_wrapper[hm3d]`（见总 README）。更换导出版本时核实下面的标定、深度类型与位姿约定。

In [ ]:
import os
from pathlib import Path
from contextlib import nullcontext
import numpy as np
from scipykit.task.cv import *
from dataset_wrapper.datasets.hm3d import HM3D
from graphmap.pose import Pose
from graphmap.sensor import CameraIntrinsic
from graphmap.adapters import rgbd_observation
from graphmap.environment import Environment

root = Path(os.environ.get("GRAPHMAP_HM3D_ROOT", "/R/Datasets/3DGS/hm3dv2"))
scene = os.environ.get("GRAPHMAP_HM3D_SCENE", "0000")
ds = HM3D(root, episodes=[scene], frame_limit=3, load_replay=True)
assert len(ds) == 3
# record() 只查看元数据；此处不读取图像。
metadata = ds.record(0).metadata
assert metadata["coordinate_frames"]["export_pose"] == "floor_relative_rgb_camera"
expected_config_hash = "7570bfa76cd07c69eee812f2b170ac0ef492a190c617e0eb897d797be7617bec"
if metadata["config_sha256"]["simulator_config"] != expected_config_hash:
    raise ValueError("此导出的配置未验证，请核对相机参数后修改此单元格")
camera = CameraIntrinsic.from_sensor(width=640, height=640, hfov=np.deg2rad(64), vfov=np.deg2rad(64))
print("scene:", scene, "sample keys:", list(ds.index), "focal:", camera.fx)

In [ ]:
first_sample = ds.iloc[0]  # 同步随机读取；也可用 ds.loc[episode, frame]
first = first_sample["data"]
fig, axes = subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(first["rgb"])
axes[0].set_title("RGB")
axes[1].imshow(first["depth"] * 0.001, cmap="viridis")
axes[1].set_title("Axial depth (m)")
if first["semantic"] is not None:
    axes[2].imshow(first["semantic"], cmap="tab20")
    axes[2].set_title("Semantic instance ID")
else:
    axes[2].text(0.5, 0.5, "No semantic labels", ha="center")
for ax in axes:
    ax.set(xlabel="w", ylabel="h")
disp(fig, '1')
plt.close(fig)

## 同一数据集切换读取方式

通过 `GRAPHMAP_HM3D_MODE` 选择 `direct`（同步按需读盘）、`prefetch`（多进程预取，默认）或 `memory`（一次预加载后从内存读取）。三者返回相同的 sample 结构，后面的建图代码完全复用。
`GRAPHMAP_HM3D_WORKERS` 默认 2。内存视图的 lambda 使用当前进程的缓存；后续读取保持在当前进程，预加载阶段可以使用多个 worker。

`stride=16` 控制建图采样量；timestamp 使用示例相对秒表示顺序，不代表原始采集时间。
没有独立 body pose 时，适配器用相机 pose 作为 Environment 的当前参考 pose。

In [ ]:
mode = os.environ.get("GRAPHMAP_HM3D_MODE", "prefetch")
num_workers = int(os.environ.get("GRAPHMAP_HM3D_WORKERS", "2"))
if mode == "direct":
    source = nullcontext(ds)  # 普通迭代在当前进程逐帧读取
elif mode == "prefetch":
    source = ds.loader(num_workers=num_workers, batch_size=None, prefetch=2)
elif mode == "memory":
    with ds.loader(num_workers=num_workers, batch_size=None, prefetch=2) as loader:
        cache_by_key = {sample["key"]: sample for sample in loader}
    memory_ds = ds.with_reader(lambda record: cache_by_key[record.key])
    source = nullcontext(memory_ds)  # 可多轮索引/迭代，不再读图像文件
else:
    raise ValueError("GRAPHMAP_HM3D_MODE 必须为 direct、prefetch 或 memory")

print("reading mode:", mode)
env = Environment(resolution=0.05, enable_2d_map=True, side_length=40)
with source as samples:
    for order, sample in enumerate(samples):
        frame = sample["data"]
        pose = sample["annotations"]["pose"]
        assert sample["metadata"]["coordinate_frames"]["export_pose"] == "floor_relative_rgb_camera"
        T_world_camera = Pose(position=pose["position"], rotation=pose["quaternion_xyzw"],
                              wrd_frame="world", ego_frame="camera")
        attributes = {} if frame["semantic"] is None else {"semantic_id": frame["semantic"]}
        observation = rgbd_observation(
            frame["depth"], intrinsic=camera, pose_camera=T_world_camera,
            rgb=frame["rgb"], attributes=attributes, depth_scale=0.001,
            depth_kind="axial", depth_range=(0.05,40), stride=16, timestamp=order * 0.1,
        )
        env.observe(**observation)
        print(sample["key"], "sampled:", len(observation["points_world"]), "map points:", env.N)
assert env.N > 0
print(env.status())

In [ ]:
semantic = env.get_index_values("semantic_id", default=-1)
counts = {int(value): semantic.count(value) for value in set(semantic)}
print("instance id -> point count:", dict(sorted(counts.items())[:12]))
if "semantic_id" in env.db:
    print("unique values:", env.index_db("semantic_id").size,
          "point links:", env.index_db("semantic_id").num_links)
# 最后一帧采样后的点/颜色/语义仍一一对齐。
h, w = observation["pixel_indices"].T
np.testing.assert_array_equal(observation["colors"], frame["rgb"][h,w])
if frame["semantic"] is not None:
    assert observation["extra_keys"]["semantic_id"].get_batch(range(len(h))) == frame["semantic"][h,w].tolist()

In [ ]:
fig = plt.figure(figsize=(11, 4), dpi=120)
ax3d = fig.add_subplot(121, projection="3d")
ax3d.scatter(*env.infopoints.xyz.T, c=env.infopoints.color[:,:3] / 255, s=2)
ax3d.set(xlabel="world x (m)", ylabel="world y (m)", zlabel="world z (m)")
axmap = fig.add_subplot(122)
axmap.imshow(env.map2d.grid[:,:,0], origin="lower", vmin=-1, vmax=1)
axmap.set(title="All observed points projected to XY", xlabel="w / world y index", ylabel="h / world x index")
# 只缩放显示窗口，不转置数组；h 对应 x，w 对应 y。
occupied_hw = env.map2d.world_to_grid(env.infopoints.xyz[:, :2])
h_min, w_min = occupied_hw.min(axis=0) - 10
h_max, w_max = occupied_hw.max(axis=0) + 10
axmap.set_xlim(w_min, w_max)
axmap.set_ylim(h_min, h_max)
fig.tight_layout()
disp(fig, '2')
plt.close(fig)

当前地图是三帧观测的投影占用示意，包含地面点；导航前应先按高度、地面判定等规则区分障碍与地面。`Environment.observe` 当前不自动清除动态物体。

新增数据源时，在 dataset_wrapper 中扩展数据集类/reader；将 sample 中的数组与已声明方向的位姿交给 `rgbd_observation`。graphmap 保持与文件格式、预取方式和缓存策略解耦。